
## Unity Catalog's Data Hierarchy

- Admins create one metastore per region
- Metastores are mapped to one or more workspaces within the same reagion.
- Metastores provide regional isolation but are not intended as units of data isolation.

## Catalogs

- Intended as prmary unit of data isolation
- Often mirror organizational units or software development lifecycle scopes.
- Can be stored at:
  - the metastore level,
  - Or eparately from the rest of the parent metastore (preferred)
  - Can be bound to speciic workspaces.
  - Ideal spot to set inherited permissions.


### Volumes

- Used for non-tabular data
- Any data - structured, semi-structured and unstructured can be stored.
- Perfect for libraries, configurations and checkpoint folders.
- Data in Volumes cannot be registered as Tables.


### Tables
  
### View 
### External Locations and Storage Credentials

- External locations are defined as a path to cloud storage, combined with a storage credential that can be used to access that location.

- Allow Unity Catalog to read and write data on user's cloud tenant.

- Use external locations to register external tables and external volumes in Unity Catalog.

- Allow Unity Catalog to read and write data on user's Cloud tenan on behalf of users.
- For enhanced data isolation, external locations and storage credentials can be bound to specific workspaces.
- To prevent bypassing Unity Catalog access controls, limit direct user access to containers used as external locations.



In [0]:
catalogo = "prd"
spark.sql(f"GRANT USE CATALOG, USE SCHEMA, SELECT ON CATALOG {catalogo} TO `account users`")

In [0]:
display(spark.sql("SHOW USERS"))

In [0]:
%sql
SHOW GRANTS ON TABLE prd.l_bronze.sm_customers

In [0]:
%sql
REVOKE USE CATALOG, USE SCHEMA, SELECT ON CATALOG prd FROM `account users`

In [0]:
%sql
SELECT * FROM email_traffic_only;


In [0]:
%sql
SELECT is_account_group_member('admins')

# Row Filtering and Column Mask

In [0]:
%sql
CREATE OR REPLACE FUNCTION erase_future_date_rows(event_timestamp STRING)
RETURNS boolean
RETURN IF(is_account_group_member('admins'), true, event_timestamp < current_timestamp());

CREATE OR REPLACE FUNCTION redact_email_traffic_only(user_id STRING)
RETURN CASE WHEN is_account_group_member('admins')
THEN user_id
ELSE '****'
END;


CREATE OR REPLACE TABLE email_traffic_only_redacted AS
SELECT * FROM email_traffic_only;

ALTER TABLE email_traffic_only_redacted
SET ROW FILTER erase_future_date_rows ON (event_timestamp);

ALTER TABLE email_traffic_only_redacted
ALTER COLUMN user_id
SET MASK redact_email_traffic_only;
    
SELECT * FROM email_traffic_only_redacted;

In [0]:
%sql
-- Table tags
ALTER TABLE email_traffic_only_redacted
SET TAGS (
  'quality'='silver',
  'domain'='customer'
)

-- Column tags
ALTER TABLE email_traffic_only_redacted
ALTER COLUMN user_id
SET TAGS (
  'pii'='email'
)



In [0]:
@dlt.table
def registered_users():
    cloud_storage_path = "dbfs:/FileStore/tables"
    return(
        spark.readStream
            .format("cloudFiles")
            .schema("user_id string, event_timestamp string, mobile boolean")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", f"{cloud_storage_path}/schema")
            .load("user_reg_source")
            )
from pyspark.sql import functions as F


def salted_hash(col):
    return F.sha2(F.concat(F.lit("salt"), col), 256)

@dlt.table
def registered_users_hashed():
    return(
        dlt.read_Stream("registered_users")
            .select(
                salted_hash(F.col("users_id")).alias("hashed_id"),
                F.col("users_id"),
                F.col("event_timestamp"),
            )
    )

@dlt.table
def registered_users_tokens():
    return(
        dlt.readStream("registered_users_hashed")
            .select("user_id")
            .distinct()
            .withColumn("token", F.expr("uuid()"))
    )
                
    